<a href="https://colab.research.google.com/github/fatmasenguler/laplacian-minor-hierarchy/blob/main/compute_chi_ijk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""1BE9_1BFE.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1415tO6UmyLh4eFEwysixqiq79Ynr8EQY
"""

# ============================================================
# χ_ijk calculation for TWO PDB files
# Uses all available residues in the PDB chain
# Does NOT stop at residue 415
# Creates separate output files for each protein
# ============================================================

import numpy as np
import pandas as pd
from scipy import linalg
import os
import re
import time
import zipfile

from google.colab import files


# ============================================================
# 0. USER SETTINGS
# ============================================================

CHAIN_ID = "A"
ATOM_TYPE = "CA"

CUTOFF = 8.0
EXCLUDE_SEQUENCE_NEIGHBORS = True

# IMPORTANT:
# Leave these as None if you want to use all residues found in the PDB file.
RESIDUE_START = None
RESIDUE_END = None

# Excel row limit is 1,048,576.
# We split output files into chunks smaller than this.
CHUNK_SIZE = 1_000_000


# ============================================================
# 1. UPLOAD TWO PDB FILES
# ============================================================

print("Upload exactly TWO PDB files:")
uploaded = files.upload()

pdb_files = [f for f in uploaded.keys() if f.lower().endswith(".pdb")]

if len(pdb_files) != 2:
    raise ValueError(f"Please upload exactly TWO .pdb files. You uploaded: {pdb_files}")

file_1 = pdb_files[0]
file_2 = pdb_files[1]

def clean_label(path):
    label = os.path.splitext(os.path.basename(path))[0]
    return re.sub(r"[^A-Za-z0-9_]+", "_", label)

label_1 = clean_label(file_1)
label_2 = clean_label(file_2)

print("\nUsing files:")
print(f"  File 1: {file_1}")
print(f"  File 2: {file_2}")
print(f"  Label 1: {label_1}")
print(f"  Label 2: {label_2}")


# ============================================================
# 2. PARSE Cα ATOMS FROM ONE CHAIN
# ============================================================

def parse_ca_chain(
    path,
    chain_id="A",
    atom_type="CA",
    residue_start=None,
    residue_end=None,
    verbose=True
):
    """
    Parse Cα atoms from a selected chain.

    If residue_start and residue_end are None, the code uses all residues
    present in the PDB file.

    Handles alternate locations:
        highest occupancy is selected.
        if tied: blank altloc > A > others.
    """

    records = {}

    with open(path, "r") as fh:
        for line in fh:
            if not line.startswith(("ATOM  ", "HETATM")):
                continue

            atom_name = line[12:16].strip()
            if atom_name != atom_type:
                continue

            chain = line[21]
            if chain != chain_id:
                continue

            try:
                resseq = int(line[22:26])
            except ValueError:
                continue

            # Optional residue filter
            if residue_start is not None and resseq < residue_start:
                continue
            if residue_end is not None and resseq > residue_end:
                continue

            icode = line[26].strip()

            altloc_raw = line[16]
            altloc = altloc_raw.strip() if altloc_raw.strip() else " "

            try:
                occupancy = float(line[54:60])
            except ValueError:
                occupancy = 0.0

            try:
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
            except ValueError:
                continue

            key = (chain, resseq, icode)

            if altloc == " ":
                altloc_priority = 2
            elif altloc == "A":
                altloc_priority = 1
            else:
                altloc_priority = 0

            priority = (occupancy, altloc_priority)

            if key not in records or priority > records[key]["priority"]:
                records[key] = {
                    "resseq": resseq,
                    "icode": icode,
                    "coords": [x, y, z],
                    "altloc": altloc,
                    "occupancy": occupancy,
                    "priority": priority,
                }

    if len(records) == 0:
        raise ValueError(
            f"No {atom_type} atoms found in chain {chain_id} of file {path}."
        )

    sorted_keys = sorted(records.keys(), key=lambda k: (k[1], k[2]))

    coords = np.array([records[k]["coords"] for k in sorted_keys], dtype=float)
    resnums = [records[k]["resseq"] for k in sorted_keys]
    icodes = [records[k]["icode"] for k in sorted_keys]
    altlocs = [records[k]["altloc"] for k in sorted_keys]

    if verbose:
        print(f"\nParsed {os.path.basename(path)}")
        print(f"  Chain              : {chain_id}")
        print(f"  Atom type          : {atom_type}")
        print(f"  First residue      : {min(resnums)}")
        print(f"  Last residue       : {max(resnums)}")
        print(f"  Number of residues : {len(resnums)}")

        full_range = set(range(min(resnums), max(resnums) + 1))
        found = set(resnums)
        missing_inside = sorted(full_range - found)

        print(f"  Missing inside {min(resnums)}–{max(resnums)}: {missing_inside}")

        nonstandard_altlocs = {
            r: a for r, a in zip(resnums, altlocs)
            if a not in (" ", "A")
        }

        if nonstandard_altlocs:
            print("  Selected non-blank/non-A altlocs:")
            print(nonstandard_altlocs)

    return coords, resnums, icodes, altlocs


# ============================================================
# 3. BUILD WEIGHTED LAPLACIAN
# ============================================================

def build_laplacian(
    coords,
    resnums,
    cutoff=8.0,
    exclude_sequence_neighbors=True
):
    """
    Build weighted Kirchhoff/Laplian matrix.

    Contact:
        d_ij < cutoff

    Weight:
        w_ij = exp(-d_ij / d_mean)

    Sequence-neighbor exclusion:
        excludes |residue_i - residue_j| <= 1
    """

    N = len(coords)

    diff = coords[:, None, :] - coords[None, :, :]
    dist = np.sqrt(np.sum(diff**2, axis=2))

    mask = (dist < cutoff) & (dist > 0.0)

    if exclude_sequence_neighbors:
        res_arr = np.array(resnums)
        seq_sep = np.abs(res_arr[:, None] - res_arr[None, :])
        mask = mask & (seq_sep > 1)

    d_contacts = dist[mask]

    if len(d_contacts) == 0:
        raise ValueError("No contacts found. Check cutoff or structure.")

    d_mean = d_contacts.mean()

    adj = np.zeros((N, N), dtype=float)
    adj[mask] = np.exp(-dist[mask] / d_mean)

    adj = 0.5 * (adj + adj.T)

    n_edges = int(np.count_nonzero(np.triu(adj, 1)))

    L = np.diag(adj.sum(axis=1)) - adj

    return L, adj, n_edges, d_mean


# ============================================================
# 4. EFFECTIVE RESISTANCE MATRIX
# ============================================================

def compute_Rij(L):
    """
    Effective resistance:

        R_ij = L^+_ii + L^+_jj - 2 L^+_ij
    """

    Lp = linalg.pinvh(L)

    diag_Lp = np.diag(Lp)
    R = diag_Lp[:, None] + diag_Lp[None, :] - 2.0 * Lp

    R = 0.5 * (R + R.T)
    np.fill_diagonal(R, 0.0)

    # Remove tiny negative numerical errors
    R = np.maximum(R, 0.0)

    return R


# ============================================================
# 5. COMPUTE χ_ijk FOR ONE PROTEIN
# ============================================================

def write_chi_for_one_protein(
    R,
    resnums,
    label,
    chunk_size=1_000_000
):
    """
    Computes χ_ijk for all ordered triples in ONE protein.

    Formula:

        K_jk^(i) = 1/2 (R_ij + R_ik - R_jk)

        χ_ijk = K^2 / (R_ij R_ik)

    Convention:

        χ_ijk = 0  uncorrelated
        χ_ijk = 1  correlated

    Output:
        i, j, k, chi_label

    The result is written directly to disk, so it works even when
    the full table is larger than Excel's row limit.
    """

    N = len(resnums)

    if N < 3:
        raise ValueError(f"{label}: need at least 3 residues for χ_ijk.")

    total_rows = N * (N - 1) * (N - 2)

    print(f"\nComputing χ_ijk for {label}")
    print(f"  Residues used : {min(resnums)}–{max(resnums)}")
    print(f"  N residues    : {N}")
    print(f"  Ordered triples: {total_rows:,}")

    full_file = f"chi_{label}_chain{CHAIN_ID}_ALL_RESIDUES_FULL.csv"

    part_files = []
    part_no = 1
    current_rows = 0
    buffer = []

    header_written_full = False
    eps = 1e-30

    all_idx = np.arange(N)
    B, C = np.meshgrid(all_idx, all_idx, indexing="ij")
    res_arr = np.array(resnums, dtype=np.int32)

    t0 = time.time()

    for a in range(N):
        if a % 10 == 0:
            print(f"  Processing i = {resnums[a]} ({a + 1}/{N})")

        mask = (B != a) & (C != a) & (B != C)

        b_idx = B[mask]
        c_idx = C[mask]

        Rab = R[a, b_idx]
        Rac = R[a, c_idx]
        Rbc = R[b_idx, c_idx]

        K = 0.5 * (Rab + Rac - Rbc)
        denom = Rab * Rac

        chi = np.full(len(b_idx), np.nan, dtype=float)
        good = denom > eps
        chi[good] = K[good] ** 2 / denom[good]

        df_a = pd.DataFrame({
            "i": res_arr[a],
            "j": res_arr[b_idx],
            "k": res_arr[c_idx],
            f"chi_{label}": chi
        })

        # Append to full CSV
        df_a.to_csv(
            full_file,
            mode="a",
            index=False,
            header=not header_written_full
        )
        header_written_full = True

        # Buffer for Excel-safe split files
        buffer.append(df_a)
        current_rows += len(df_a)

        if current_rows >= chunk_size:
            df_part = pd.concat(buffer, ignore_index=True)
            part_file = f"chi_{label}_chain{CHAIN_ID}_ALL_RESIDUES_part{part_no}.csv"
            df_part.to_csv(part_file, index=False)
            part_files.append(part_file)

            print(f"  Saved {part_file} with {len(df_part):,} rows")

            part_no += 1
            buffer = []
            current_rows = 0

    # Save remaining rows
    if buffer:
        df_part = pd.concat(buffer, ignore_index=True)
        part_file = f"chi_{label}_chain{CHAIN_ID}_ALL_RESIDUES_part{part_no}.csv"
        df_part.to_csv(part_file, index=False)
        part_files.append(part_file)

        print(f"  Saved {part_file} with {len(df_part):,} rows")

    elapsed = time.time() - t0

    print(f"\nFinished {label}")
    print(f"  Time: {elapsed:.1f} seconds")
    print(f"  Full file: {full_file}")
    print(f"  Split files: {part_files}")

    return full_file, part_files, total_rows


# ============================================================
# 6. ANALYZE ONE PROTEIN
# ============================================================

def analyze_one_protein(path, label):
    coords, resnums, icodes, altlocs = parse_ca_chain(
        path,
        chain_id=CHAIN_ID,
        atom_type=ATOM_TYPE,
        residue_start=RESIDUE_START,
        residue_end=RESIDUE_END,
        verbose=True
    )

    L, adj, n_edges, d_mean = build_laplacian(
        coords,
        resnums,
        cutoff=CUTOFF,
        exclude_sequence_neighbors=EXCLUDE_SEQUENCE_NEIGHBORS
    )

    print(f"\nContact network for {label}")
    print(f"  Number of residues : {len(resnums)}")
    print(f"  Number of edges    : {n_edges}")
    print(f"  Mean contact dist. : {d_mean:.6f} Å")

    R = compute_Rij(L)

    full_file, part_files, total_rows = write_chi_for_one_protein(
        R,
        resnums,
        label,
        chunk_size=CHUNK_SIZE
    )

    summary_file = f"summary_{label}_chain{CHAIN_ID}_ALL_RESIDUES.txt"

    full_range = set(range(min(resnums), max(resnums) + 1))
    missing_inside = sorted(full_range - set(resnums))

    with open(summary_file, "w") as f:
        f.write("χ_ijk calculation summary\n")
        f.write("=========================\n\n")

        f.write(f"File: {path}\n")
        f.write(f"Label: {label}\n")
        f.write(f"Chain: {CHAIN_ID}\n")
        f.write(f"Atom type: {ATOM_TYPE}\n")
        f.write(f"Cutoff: {CUTOFF} Å\n")
        f.write(f"Exclude sequence neighbors: {EXCLUDE_SEQUENCE_NEIGHBORS}\n\n")

        f.write("Residue selection:\n")
        f.write("  Used all residues found in the PDB chain.\n")
        f.write(f"  First residue: {min(resnums)}\n")
        f.write(f"  Last residue: {max(resnums)}\n")
        f.write(f"  Number of residues: {len(resnums)}\n")
        f.write(f"  Missing inside range: {missing_inside}\n\n")

        f.write("Contact network:\n")
        f.write(f"  Number of edges: {n_edges}\n")
        f.write(f"  Mean contact distance: {d_mean:.6f} Å\n\n")

        f.write("χ_ijk output:\n")
        f.write(f"  Total ordered triples: {total_rows}\n")
        f.write(f"  Expected N(N-1)(N-2): {len(resnums) * (len(resnums)-1) * (len(resnums)-2)}\n")
        f.write(f"  Full CSV: {full_file}\n")
        f.write(f"  Split CSV files: {part_files}\n")

    print(f"  Summary file: {summary_file}")

    return {
        "label": label,
        "path": path,
        "resnums": resnums,
        "full_file": full_file,
        "part_files": part_files,
        "summary_file": summary_file,
        "n_edges": n_edges,
        "d_mean": d_mean,
        "total_rows": total_rows
    }


# ============================================================
# 7. RUN BOTH PROTEINS SEPARATELY
# ============================================================

result_1 = analyze_one_protein(file_1, label_1)
result_2 = analyze_one_protein(file_2, label_2)


# ============================================================
# 8. SAVE GLOBAL SUMMARY
# ============================================================

global_summary = "summary_TWO_PROTEINS_ALL_RESIDUES.txt"

with open(global_summary, "w") as f:
    f.write("Two-protein χ_ijk calculation summary\n")
    f.write("====================================\n\n")

    for result in [result_1, result_2]:
        resnums = result["resnums"]

        f.write(f"Protein: {result['label']}\n")
        f.write(f"  File: {result['path']}\n")
        f.write(f"  First residue: {min(resnums)}\n")
        f.write(f"  Last residue: {max(resnums)}\n")
        f.write(f"  Number of residues: {len(resnums)}\n")
        f.write(f"  Number of edges: {result['n_edges']}\n")
        f.write(f"  d_mean: {result['d_mean']:.6f} Å\n")
        f.write(f"  Total χ_ijk rows: {result['total_rows']}\n")
        f.write(f"  Full file: {result['full_file']}\n")
        f.write(f"  Split files: {result['part_files']}\n\n")

print(f"\nSaved global summary: {global_summary}")


# ============================================================
# 9. ZIP AND DOWNLOAD EVERYTHING
# ============================================================

zip_name = f"chi_TWO_PROTEINS_chain{CHAIN_ID}_ALL_RESIDUES_outputs.zip"

all_output_files = [
    result_1["full_file"],
    result_1["summary_file"],
    result_2["full_file"],
    result_2["summary_file"],
    global_summary,
]

all_output_files += result_1["part_files"]
all_output_files += result_2["part_files"]

with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in all_output_files:
        if os.path.exists(fname):
            zf.write(fname)

print("\nCreated ZIP file:")
print(f"  {zip_name}")

files.download(zip_name)

Upload exactly TWO PDB files:


Saving 1BE9.pdb to 1BE9.pdb
Saving 1BFE.pdb to 1BFE.pdb

Using files:
  File 1: 1BE9.pdb
  File 2: 1BFE.pdb
  Label 1: 1BE9
  Label 2: 1BFE

Parsed 1BE9.pdb
  Chain              : A
  Atom type          : CA
  First residue      : 301
  Last residue       : 415
  Number of residues : 115
  Missing inside 301–415: []

Contact network for 1BE9
  Number of residues : 115
  Number of edges    : 431
  Mean contact dist. : 6.249491 Å

Computing χ_ijk for 1BE9
  Residues used : 301–415
  N residues    : 115
  Ordered triples: 1,481,430
  Processing i = 301 (1/115)
  Processing i = 311 (11/115)
  Processing i = 321 (21/115)
  Processing i = 331 (31/115)
  Processing i = 341 (41/115)
  Processing i = 351 (51/115)
  Processing i = 361 (61/115)
  Processing i = 371 (71/115)
  Saved chi_1BE9_chainA_ALL_RESIDUES_part1.csv with 1,004,796 rows
  Processing i = 381 (81/115)
  Processing i = 391 (91/115)
  Processing i = 401 (101/115)
  Processing i = 411 (111/115)
  Saved chi_1BE9_chainA_ALL_RESIDUES_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>